In [ ]:
from ultralytics import YOLO
from collections import Counter
import os
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

## Data Processing

In [ ]:
def load_class_names(label_folder):
    classes_file = os.path.join(label_folder, "classes.txt")
    if not os.path.exists(classes_file):
        print(f"{classes_file} file not exists!")
        return {}

    with open(classes_file, "r", encoding="utf-8") as f:
        class_names = [line.strip() for line in f.readlines()]
    
    return {i: class_names[i] for i in range(len(class_names))}

def count_labels_and_negatives(label_folder):
    category_count = Counter()
    negative_samples = 0

    for label_file in os.listdir(label_folder):
        if label_file.endswith(".txt") and label_file != "classes.txt":
            file_path = os.path.join(label_folder, label_file)
            with open(file_path, "r", encoding="utf-8") as f:
                lines = [line.strip() for line in f.readlines() if line.strip()]
                if not lines:
                    negative_samples += 1 
                else:
                    for line in lines:
                        class_id = int(line.split()[0])
                        category_count[class_id] += 1

    return category_count, negative_samples


The number of labels (per class)

In [ ]:
datasets = ["train", "test", "val"]

# DATA PATH
project_dir = Path(os.getcwd()).parent
dataset_path = os.path.join(project_dir,"Dataset",'Mars')# For MARS or For MOON

base_path = os.path.join(dataset_path,'labels')

for dataset in datasets:
    label_folder = os.path.join(base_path, dataset)
    class_map = load_class_names(label_folder)
    counts, negative_samples = count_labels_and_negatives(label_folder)

    print(f"\n {dataset.upper()} dataset:")
    
    for class_id in sorted(class_map.keys()):
        class_name = class_map.get(class_id, f"class {class_id}")
        count = counts.get(class_id, 0)
        print(f"   {class_name} (class {class_id}): {count}")

Visualizes YOLO-format bounding boxes on images by reading labels, converting coordinates, and drawing colored boxes with class names. It processes the first 2 images from the train subset.

In [ ]:
# Draw bounding boxes on images based on YOLO labels.
def visualize_annotations(image_path, label_path, class_map, class_colors):
    image = cv2.imread(image_path)
    h, w, _ = image.shape

    if not os.path.exists(label_path):
        print(f"No label found for {image_path}")
        return
    
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            class_id = int(parts[0])
            x_center, y_center, box_w, box_h = map(float, parts[1:])
            
            x1 = int((x_center - box_w / 2) * w)
            y1 = int((y_center - box_h / 2) * h)
            x2 = int((x_center + box_w / 2) * w)
            y2 = int((y_center + box_h / 2) * h)

            label = class_map.get(class_id, f"Class {class_id}")
            color = class_colors.get(class_id, (255, 255, 255))

            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            cv2.putText(image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

In [ ]:
# Define dataset path
subset = "train"
image_folder = os.path.join(dataset_path, "images", subset)
label_folder = os.path.join(dataset_path, "labels", subset)

# Load class names
class_map = load_class_names(label_folder)

# Assign distinct colors to each class
class_colors = {
    0: (255, 0, 0),   # Large crater - Blue
    1: (0, 255, 0),   # Small crater - Green
    2: (0, 0, 255),   # Medium crater - RED
}

# Get a list of image files (limit to first 3)
image_files = [f for f in os.listdir(image_folder) if f.endswith(".jpg")][:2]

if not image_files:
    print("No images found in the dataset!")
else:
    for image_file in image_files:
        image_path = os.path.join(image_folder, image_file)
        label_path = os.path.join(label_folder, image_file.replace(".jpg", ".txt"))
        visualize_annotations(image_path, label_path, class_map, class_colors)


Visualizes the distribution of bounding box areas (width × height) for crater classes (Large/Small/Medium) across train/val/test datasets

In [ ]:
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly"

class_name_map = {
    0: "Large Crater",
    1: "Small Crater",
    2: "Medium Crater"
}

color_map = {
    "Large Crater": "orange",
    "Small Crater": "blue",
    "Medium Crater": "red",
}

class_order = ["Large Crater", "Small Crater", "Medium Crater"]

def get_bounding_box_data(label_folder):
    data = []
    for label_file in os.listdir(label_folder):
        if label_file.endswith(".txt") and label_file != "classes.txt":
            with open(os.path.join(label_folder, label_file), "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        box_w, box_h = map(float, parts[3:5])
                        area = box_w * box_h
                        data.append({
                            "class": class_name_map[class_id],
                            "area": area
                        })
    return pd.DataFrame(data)

def plot_bounding_box_distribution(labels_df, title):
    if labels_df.empty:
        print(f"No bounding box data available for {title}")
        return

    fig = px.histogram(
        labels_df,
        x='area',
        nbins=50,
        color='class',
        category_orders={"class": class_order},
        color_discrete_map=color_map,
        title=title
    )
    fig.update_layout(barmode='overlay')
    fig.update_traces(opacity=0.6)
    fig.update_yaxes(type="log")
    fig.show()

YOLO

In [ ]:
#import torch_directml
import torch
import yaml

dataset_path = Path(dataset_path)
# create .YAML
data_config = {
    "path": str(dataset_path.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 3,
    "names": ["Large crater", "Small crater", "Medium crater"]
}
# save .YAML
data_yaml = "craters_Mars.yaml"
with open(data_yaml, "w") as f:
    yaml.dump(data_config, f)

In [ ]:
import sys
from ultralytics import YOLO

# sys.stdout = open('training_log.txt', 'w')
# print("THIS IS A TEST") 

model = YOLO("yolo11n.pt")
model.train(
    data=data_yaml,
    epochs=200,
    imgsz=600,
    batch=16,
    workers=3,
    verbose=False,
    cache=True,
    optimizer="Adam",
    cos_lr=True,
    project="my_training", 
    name="exp1"
    )

In [ ]:
#Curves - change to Mars or Moon versions
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("my_training/exp1/results.csv")
metric_report_train = {
    'train/box_loss': 'Box Loss',
    'train/cls_loss': 'Cls Loss',
    'train/dfl_loss': 'Dfl Loss',
}
metric_report_val = {
    'val/box_loss': 'Box Loss',
    'val/cls_loss': 'Cls Loss',
    'val/dfl_loss': 'Dfl Loss'
}
metric_report_map = {
    'metrics/mAP50-95(B)': 'mAP@0.50-0.95'
}

def metric_report(df, metrics_def, y_label, file):
    fig, ax = plt.subplots()
    for metric, label in metrics_def.items():
        ax.plot(df['epoch'], df[metric], label = label)
        
    ax.set_xlabel('Epoch', fontsize = 14)
    ax.set_ylabel(f'{y_label}', fontsize = 14)
    ax.tick_params(
        axis = 'both',
        labelsize = 14,
        length = 6,
        width = 1.5
    )
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{file} YOLO Mars.png')
    plt.close()

metric_report(df, metric_report_train, 'Loss', 'Training Losses')
metric_report(df, metric_report_val, 'Loss', 'Val Losses')
metric_report(df, metric_report_map, 'mAP', 'mAP50-95')

In [ ]:
model = YOLO('my_training/exp1/weights/best.pt')

results = model.val(data=data_yaml, 
                    split='test', 
                    conf=0.001, 
                    iou=0.6, 
                    save_json=True,
                    save_conf=True,
                    verbose=False
)

In [ ]:
print(results.box.curves_results[0][1]) #precisions for all classes
precisions = results.box.curves_results[0][1]
mean_recall = results.box.curves_results[0][0]
recall = results.box.curves_results[0][0]
aP50 = results.box.ap50

for i in range(3):
    precision = precisions[i, :]
    print(precision)
    plt.plot(recall, precision, lw=2, label= f'Class {i} {aP50[i]:.3f}')

mean_precision = np.mean(precisions, axis=0)
mAP = np.mean(aP50)
plt.plot(mean_recall, mean_precision, lw = 2, label = f'All Classes {mAP:.3f} mAP@0.5')
plt.xlabel("Recall", fontsize=14)
plt.ylabel("Precision", fontsize=14)
plt.tick_params(
    axis='both',
    labelsize=14,
    length = 6,
    width = 1.5
)
plt.legend(loc="best", fontsize=12)
# TODO：For MARS or For MOON
plt.tight_layout()
plt.savefig(f"PR Curves of YOLO Mars.png")
plt.close()

In [ ]:
num_classes = 3
num_experiments = 1
experiment_records = {
    'f1': np.zeros((num_experiments, num_classes)),
    'precision': np.zeros((num_experiments, num_classes)),
    'recall': np.zeros((num_experiments, num_classes))
}

model = YOLO('my_training/exp1/weights/best.pt')
# results = model.val(
#     data=data_yaml,
#     split = 'test',
#     conf=0.001,
#     iou=0.6,
#     verbose=False
# )

# experiment_records['f1'][0] = results.box.f1
# experiment_records['precision'][0] = results.box.p
# experiment_records['recall'][0] = results.box.r


for exp_id in range(num_experiments):
    results = model.val(data=data_yaml, split='test', conf=0.001, iou=0.6, verbose=False)
    experiment_records['f1'][exp_id] = results.box.f1
    experiment_records['precision'][exp_id] = results.box.p
    experiment_records['recall'][exp_id] = results.box.r

final_metrics = {
    metric: np.column_stack((
        experiment_records[metric].mean(axis=0),
        experiment_records[metric].std(axis=0)
    )) for metric in ['f1', 'precision', 'recall']
}

print(final_metrics)

In [ ]:
MODEL_PATH = "my_training/exp1/weights/best.pt"
IMAGE_PATH = "resized_4k.jpg"
OUTPUT_PATH = "Predicted_1.jpg"
CROP_SIZE = 960
STRIDE_RATIO = 0.3
MIN_CONFIDENCE = 0.5
NMS_IOU = 0.4
DEBUG_MODE = False

CLASS_COLORS = {
    0: (255, 0, 0),
    1: (0, 255, 0),
    2: (0, 0, 255)
}

CLASS_NAMES = {
    0: "Large Crater",
    1: "Small Crater",
    2: "Medium Crater"
}

def intelligent_crop(img, crop_size, stride_ratio):
    h, w = img.shape[:2]
    stride = int(crop_size * (1 - stride_ratio))
    
    for y in range(0, h - crop_size + 1, stride):
        for x in range(0, w - crop_size + 1, stride):
            actual_x = min(x, w - crop_size)
            actual_y = min(y, h - crop_size)
            yield img[actual_y:actual_y+crop_size, actual_x:actual_x+crop_size], (actual_x, actual_y)

def class_aware_nms(detections, iou_threshold):
    from collections import defaultdict
    import torch
    from torchvision.ops import nms
    
    class_groups = defaultdict(list)
    for det in detections:
        class_groups[det[1]].append(det)
    
    keep = []
    for cls, group in class_groups.items():
        boxes = torch.tensor([d[0] for d in group], dtype=torch.float32)
        scores = torch.tensor([d[2] for d in group], dtype=torch.float32)
        indices = nms(boxes, scores, iou_threshold)
        keep.extend([group[i] for i in indices])
    return keep

def validate_overlaps(detections):
    validated = []
    for box, cls, conf in detections:
        conflict = False
        for v_box, v_cls, v_conf in validated:
            if iou(box, v_box) > 0.7 and cls != v_cls:
                if conf > v_conf:
                    validated.remove((v_box, v_cls, v_conf))
                else:
                    conflict = True
                    break
        if not conflict:
            validated.append((box, cls, conf))
    return validated

def iou(box1, box2):
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    
    inter_xmin = max(x1_min, x2_min)
    inter_ymin = max(y1_min, y2_min)
    inter_xmax = min(x1_max, x2_max)
    inter_ymax = min(y1_max, y2_max)
    
    inter_area = max(0.0, inter_xmax - inter_xmin) * max(0.0, inter_ymax - inter_ymin)
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    
    return inter_area / (box1_area + box2_area - inter_area + 1e-6)

def main():
    model = YOLO(MODEL_PATH)
    orig_img = cv2.imread(IMAGE_PATH)
    h, w = orig_img.shape[:2]
    
    all_detections = []
    
    for crop, (x_offset, y_offset) in intelligent_crop(orig_img, CROP_SIZE, STRIDE_RATIO):
        results = model(crop, conf=MIN_CONFIDENCE, verbose=False)
        
        for result in results:
            for box in result.boxes:
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                abs_coords = (
                    float(x1 + x_offset), 
                    float(y1 + y_offset),
                    float(x2 + x_offset), 
                    float(y2 + y_offset)
                )
                all_detections.append((abs_coords, cls_id, conf))
        
        if DEBUG_MODE:
            debug_img = crop.copy()
            cv2.rectangle(debug_img, (0,0), (CROP_SIZE,CROP_SIZE), (0,255,255), 2)
            cv2.imshow('Window Debug', debug_img)
            cv2.waitKey(1)
    
    final_detections = []
    if all_detections:
        nms_detections = class_aware_nms(all_detections, NMS_IOU)
        final_detections = validate_overlaps(nms_detections)
    
    output_img = orig_img.copy()
    class_counts = Counter()
    
    for box, cls_id, conf in final_detections:
        class_counts[cls_id] += 1
        
        int_box = tuple(map(int, box))
        color = CLASS_COLORS.get(cls_id, (255, 255, 255))
        label = f"{CLASS_NAMES.get(cls_id, 'Unknown')} {conf:.2f}"
        
        cv2.rectangle(output_img, (int_box[0], int_box[1]), 
                     (int_box[2], int_box[3]), color, 2)
        cv2.putText(output_img, label, (int_box[0], int_box[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    cv2.imwrite(OUTPUT_PATH, output_img)
    
    print("\n Statistics Information：")
    for cls_id, count in sorted(class_counts.items()):
        class_name = CLASS_NAMES.get(cls_id, f"Unknown Class {cls_id}")
        print(f" {class_name}: {count}")

if __name__ == "__main__":
    main()


In [ ]:
class_colors = {
    0: (255, 0, 0),
    1: (0, 255, 0),
    2: (0, 0, 255)
}

class_names = {
    0: "Large Crater",
    1: "Small Crater",
    2: "Medium Crater"
}

model = YOLO("my_training/exp1/weights/best.pt")

resized_image_path = "resized_4k.jpg"
image = cv2.imread(resized_image_path)

TARGET_WIDTH = 3840  

results_full = model.predict(resized_image_path, imgsz=TARGET_WIDTH)[0]

detections_full = []
class_counts = Counter()

for box in results_full.boxes:
    xyxy = box.xyxy[0].cpu().numpy()
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])

    detections_full.append((xyxy, class_id, confidence))
    class_counts[class_id] += 1

output_image = image.copy()

for det, class_id, confidence in detections_full:
    x1, y1, x2, y2 = map(int, det)
    color = class_colors.get(class_id, (255, 255, 255))

    box_width = x2 - x1
    font_scale = max(0.5, box_width / 200)
    font_thickness = max(1, int(font_scale * 2))

    cv2.rectangle(output_image, (x1, y1), (x2, y2), color, 2)

    class_name = class_names.get(class_id, f"Unknown {class_id}")
    label = f"{class_name} {confidence:.2f}"

    text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)[0]
    text_x, text_y = x1, y1 - 10

    if text_y < text_size[1]:
        text_y = y1 + text_size[1] + 10

    cv2.putText(output_image, label, (text_x, text_y), 
                cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, font_thickness)

output_path = "Predicted_2.jpg"
cv2.imwrite(output_path, output_image)

print("\n Statistics Information：")
for class_id, count in sorted(class_counts.items()):
    class_name = class_names.get(class_id, f"Unknown class {class_id}")
    print(f" {class_name}: {count}")
